# Statistics and Reports

## Import Libraries

In [1]:
import base64
import tkinter as tk
from types import SimpleNamespace

import pandas as pd
import plotly.express as px
import ttkbootstrap as ttk

from ipynb.fs.full.base_frame import BaseFrame

## Statistics and Reports Class

In [2]:
# noinspection PyTypeChecker
class StatisticsReports(BaseFrame):
    """Render statistical metrics and interactive charts for Digital Library data.

    This frame receives shared references from ``DigitalLibrary`` and builds a
    chart-only dashboard with four metrics:
    1) Loan quantity by user type
    2) Book quantity by category
    3) Most loaned books
    4) Top 5 users by loan count

    The class supports two execution modes:
    - UI mode: builds Tkinter containers and renders chart images in labels.
    - ``no_ui=True`` mode: keeps pure data/chart logic available for unit tests.
    """

    def __init__(self, master=None, shared_categories=None, shared_users=None, shared_loans=None, no_ui=False):
        """Initialize shared references, UI caches, and optional frame widgets.

        Args:
            master: Parent Tkinter widget used when UI mode is enabled.
            shared_categories: Shared ``dict[str, list[Book]]`` from DigitalLibrary.
            shared_users: Shared ``dict[str, list[User]]`` from DigitalLibrary.
            shared_loans: Shared ``list[dict]`` of loan records.
            no_ui: When True, skip BaseFrame initialization for headless testing.
        """
        self.categories = shared_categories if shared_categories is not None else {}
        self.users = shared_users if shared_users is not None else {}
        self.loans = shared_loans if shared_loans is not None else []

        self._chart_labels = {}
        self._chart_images = {}
        self._render_job_id = None

        if not no_ui:
            super().__init__(master)

    def get_title(self):
        """Return the feature title shown by the base frame header."""
        return "📈 Statistics and Reports"

    def build_right_panel(self) -> None:
        """Build action controls in the right panel.

        The right panel contains a single action button used to refresh all charts.
        """
        super().build_right_panel()
        actions = ttk.Frame(master=self.right_panel, padding=(10, 8))
        actions.pack(fill=tk.X)

        ttk.Button(master=actions, text="Update Charts", command=self.request_chart_update).pack(fill=tk.X)

    def build_left_panel(self) -> None:
        """Build the left panel with a 2x2 chart grid and metric title label.

        Creates chart placeholders that initially show loading text, then schedules
        asynchronous chart rendering to avoid blocking initial UI paint.
        """
        super().build_left_panel()
        self.loading_label = ttk.Label(
            master=self.left_panel_container,
            text="Digital Library Metrics",
            font=("Helvetica", 14, "bold")
        )
        self.loading_label.grid(row=0, column=0, columnspan=2, sticky=tk.W, padx=10, pady=(10, 6))
        self.left_panel_container.grid_columnconfigure(0, weight=1)
        self.left_panel_container.grid_columnconfigure(1, weight=1)
        self.left_panel_container.grid_rowconfigure(1, weight=1)
        self.left_panel_container.grid_rowconfigure(2, weight=1)

        chart_titles = [
            ("loans_by_type", "Loan Quantity by User Type"),
            ("books_by_category", "Book Quantity by Category"),
            ("most_loaned_books", "Most Loaned Books"),
            ("top_users", "Top 5 Users by Loans")
        ]
        for idx, (chart_key, title) in enumerate(chart_titles):
            row = 1 + (idx // 2)
            column = idx % 2
            frame = ttk.Labelframe(master=self.left_panel_container, text=title, padding=(8, 8))
            frame.grid(row=row, column=column, sticky=tk.NSEW, padx=10, pady=(0, 8))
            frame.grid_rowconfigure(0, weight=1)
            frame.grid_columnconfigure(0, weight=1)
            label = ttk.Label(
                master=frame,
                text="Loading! Please wait...",
                anchor=tk.CENTER,
                font=("Helvetica", 24, "bold")
            )
            label.grid(row=0, column=0, sticky=tk.NSEW)
            self._chart_labels[chart_key] = label

        self.request_chart_update()

    def request_chart_update(self) -> None:
        """Queue chart rendering with ``after`` so loading labels render first.

        This method resets chart placeholders to loading state, cancels any pending
        rendering job, and schedules a new render task.
        """
        if not self._chart_labels:
            return
        for label in self._chart_labels.values():
            label.configure(
                text="Loading! Please wait...",
                image="",
                font=("Helvetica", 24, "bold")
            )
        self.left_panel_container.update_idletasks()
        if self._render_job_id is not None:
            self.left_panel_container.after_cancel(self._render_job_id)
        self._render_job_id = self.left_panel_container.after(50, self._render_charts_job)

    def _render_charts_job(self) -> None:
        """Execute a scheduled chart refresh job.

        Clears the pending job id and updates all chart images from current data.
        """
        self._render_job_id = None
        self.update_all_charts()

    def _build_user_lookup(self):
        """Build lookup structures for user metadata.

        Returns:
            tuple: ``(email_to_name, email_to_type, known_types)`` where:
            - ``email_to_name`` maps normalized email to user display name.
            - ``email_to_type`` maps normalized email to user type label.
            - ``known_types`` is the set of user type labels present in users data.
        """
        email_to_name = {}
        email_to_type = {}
        known_types = set()

        for users_group in self.users.values():
            for user in users_group:
                email = str(getattr(user, "email", "")).strip().lower()
                if not email:
                    continue
                name = str(getattr(user, "name", "")).strip()
                user_type = str(getattr(getattr(user, "user_type", SimpleNamespace(label="")), "label", "")).strip()
                if not user_type:
                    continue
                email_to_name[email] = name
                email_to_type[email] = user_type
                known_types.add(user_type)

        return email_to_name, email_to_type, known_types

    def _prepare_loans_by_user_type_data(self) -> dict:
        """Prepare aggregated counts for the 'Loan Quantity by User Type' chart.

        Returns:
            dict[str, int]: User-type label mapped to number of associated loans.
        """
        _email_to_name, email_to_type, known_types = self._build_user_lookup()
        counts = {user_type: 0 for user_type in known_types}

        for loan in self.loans:
            email = str(loan.get("user_email", "")).strip().lower()
            user_type = email_to_type.get(email, "")
            if not user_type:
                continue
            if user_type not in counts:
                continue
            counts[user_type] += 1
        return counts

    def _prepare_book_quantity_by_category_data(self) -> dict:
        """Prepare per-category counts of distinct book records.

        Each book is identified by ``(title, isbn)`` in its category.

        Returns:
            dict[str, int]: Category mapped to number of distinct books.
        """
        totals = {}
        for category, books in self.categories.items():
            seen_titles = set()
            for book in books:
                title = str(getattr(book, "title", "")).strip()
                isbn = str(getattr(book, "isbn", "")).strip()
                key = (title, isbn)
                if not title and not isbn:
                    continue
                seen_titles.add(key)
            totals[category] = len(seen_titles)
        return totals

    def _prepare_most_loaned_books_data(self) -> dict:
        """Prepare ranked counts for most-loaned books.

        Returns:
            dict[str, int]: Book title mapped to loan count, ordered descending.
        """
        book_counts = {}
        seen = set()
        for loan in self.loans:
            title = str(loan.get("book_title", "")).strip()
            isbn = str(loan.get("isbn", "")).strip()
            if not title or not isbn:
                continue
            key = (title, isbn)
            seen.add(key)
            if key not in book_counts:
                book_counts[key] = 0
            book_counts[key] += 1

        ordered = sorted(book_counts.items(), key=lambda item: item[1], reverse=True)
        result = {}
        for (title, isbn), count in ordered:
            result[f"{title}"] = count
        return result

    def _prepare_top_users_data(self, limit: int = 5) -> dict:
        """Prepare ranked counts for top users by loan quantity.

        Args:
            limit: Maximum number of users to include in descending ranking.

        Returns:
            dict[str, int]: User name mapped to loan count.
        """
        email_to_name, _email_to_type, _known_types = self._build_user_lookup()
        user_counts = {}
        seen_emails = set()

        for loan in self.loans:
            email = str(loan.get("user_email", "")).strip().lower()
            if not email or email not in email_to_name:
                continue
            seen_emails.add(email)
            if email not in user_counts:
                user_counts[email] = 0
            user_counts[email] += 1

        ordered = sorted(user_counts.items(), key=lambda item: item[1], reverse=True)[:limit]
        result = {}
        for email, count in ordered:
            name = email_to_name.get(email, "")
            if not name:
                continue
            result[f"{name}"] = count
        return result

    @staticmethod
    def _dict_to_dataframe(data: dict, label_col: str, value_col: str) -> pd.DataFrame:
        """Convert a mapping into a two-column DataFrame.

        Args:
            data: Source mapping used to generate rows.
            label_col: Output column name for mapping keys.
            value_col: Output column name for mapping values.

        Returns:
            pd.DataFrame: DataFrame with ``[label_col, value_col]`` columns.
        """
        rows = [{label_col: key, value_col: value} for key, value in data.items()]
        return pd.DataFrame(rows, columns=[label_col, value_col])

    def build_chart_loans_by_user_type(self):
        """Build Plotly bar chart for loans aggregated by user type.

        Returns:
            plotly.graph_objects.Figure: Configured bar chart figure.
        """
        data = self._prepare_loans_by_user_type_data()
        df = self._dict_to_dataframe(data, "User Type", "Loans")
        return px.bar(df, x="User Type", y="Loans", title="Loan Quantity by User Type", text="Loans")

    def build_chart_book_quantity_by_category(self):
        """Build Plotly bar chart for distinct books per category.

        Returns:
            plotly.graph_objects.Figure: Configured bar chart figure.
        """
        data = self._prepare_book_quantity_by_category_data()
        df = self._dict_to_dataframe(data, "Category", "Quantity")
        return px.bar(df, x="Category", y="Quantity", title="Book Quantity by Category", text="Quantity")

    def build_chart_most_loaned_books(self):
        """Build Plotly bar chart for most-loaned books.

        Returns:
            plotly.graph_objects.Figure: Configured bar chart figure.
        """
        data = self._prepare_most_loaned_books_data()
        df = self._dict_to_dataframe(data, "Book", "Loans")
        return px.bar(df, x="Book", y="Loans", title="Most Loaned Books", text="Loans")

    def build_chart_top_5_users(self):
        """Build Plotly bar chart for top 5 users by loan count.

        Returns:
            plotly.graph_objects.Figure: Configured bar chart figure.
        """
        data = self._prepare_top_users_data(limit=5)
        df = self._dict_to_dataframe(data, "User", "Loans")
        return px.bar(df, x="User", y="Loans", title="Top 5 Users by Loan Quantity", text="Loans")

    @staticmethod
    def _safe_fig_to_photo(fig):
        """Convert a Plotly figure into a Tk-compatible image.

        Uses ``fig.to_image`` and returns ``None`` when image export is not
        available (for example, when Kaleido is not installed).

        Args:
            fig: Plotly figure instance.

        Returns:
            tk.PhotoImage | None: Decoded image for Tk labels, or ``None``.
        """
        try:
            image_bytes = fig.to_image(format="png", width=600, height=340, scale=1)
            encoded = base64.b64encode(image_bytes).decode("ascii")
            return tk.PhotoImage(data=encoded)
        except Exception:
            return None

    def _set_chart_image(self, chart_key: str, fig) -> None:
        """Render one chart figure into its mapped label widget.

        Args:
            chart_key: Internal chart identifier in ``self._chart_labels``.
            fig: Plotly figure to render in the target label.
        """
        label = self._chart_labels.get(chart_key)
        if label is None:
            return
        photo = self._safe_fig_to_photo(fig)
        if photo is None:
            label.configure(text="Unable to render chart image (install 'kaleido').", image="")
            return
        self._chart_images[chart_key] = photo
        label.configure(image=photo, text="")

    def update_all_charts(self) -> None:
        """Refresh all chart panels using current shared data."""
        if not self._chart_labels:
            return
        self._set_chart_image("loans_by_type", self.build_chart_loans_by_user_type())
        self._set_chart_image("books_by_category", self.build_chart_book_quantity_by_category())
        self._set_chart_image("most_loaned_books", self.build_chart_most_loaned_books())
        self._set_chart_image("top_users", self.build_chart_top_5_users())

    def load_data(self) -> None:
        """Compatibility hook for BaseFrame API.

        Data is consumed directly from shared references; no standalone load step
        is required in this class.
        """
        return None


## Statistics Reports Tests

In [3]:
import unittest


class StatisticsReportsTest(unittest.TestCase):
    """Unit tests for StatisticsReports data preparation and chart builders."""

    def _user(self, name, email, label):
        """Create a minimal user-like object for test datasets."""
        return SimpleNamespace(name=name, email=email, user_type=SimpleNamespace(label=label), user_id=email)

    def _book(self, title, isbn, quantity, author="A"):
        """Create a minimal book-like object for test datasets."""
        return SimpleNamespace(title=title, isbn=isbn, quantity=quantity, author=author)

    def _build_report(self):
        """Create a headless StatisticsReports instance with sample shared data."""
        users = {
            "Student": [self._user("Ana", "ana@mail.com", "Student")],
            "Professor": [self._user("Bruno", "bruno@mail.com", "Professor")]
        }
        categories = {
            "Fiction": [self._book("Dune", "1", 4), self._book("Hobbit", "2", 3)],
            "Technology": [self._book("Clean Code", "3", 5)]
        }
        loans = [
            {"user_email": "ana@mail.com", "book_title": "Dune", "isbn": "1"},
            {"user_email": "ana@mail.com", "book_title": "Dune", "isbn": "1"},
            {"user_email": "bruno@mail.com", "book_title": "Clean Code", "isbn": "3"}
        ]
        return StatisticsReports(
            shared_categories=categories,
            shared_users=users,
            shared_loans=loans,
            no_ui=True
        )

    def test_prepare_loans_by_user_type(self):
        """Validate loan aggregation by user type."""
        report = self._build_report()
        data = report._prepare_loans_by_user_type_data()
        self.assertEqual(data["Student"], 2)
        self.assertEqual(data["Professor"], 1)

    def test_prepare_book_quantity_by_category(self):
        """Validate distinct book counting by category."""
        report = self._build_report()
        data = report._prepare_book_quantity_by_category_data()
        self.assertEqual(data["Fiction"], 2)
        self.assertEqual(data["Technology"], 1)

    def test_prepare_most_loaned_books(self):
        """Validate ranking/counting for most-loaned books."""
        report = self._build_report()
        data = report._prepare_most_loaned_books_data()
        self.assertEqual(data["Dune"], 2)

    def test_prepare_top_users(self):
        """Validate top user ranking by number of loans."""
        report = self._build_report()
        data = report._prepare_top_users_data(limit=5)
        self.assertEqual(data["Ana"], 2)

    def test_build_chart_objects(self):
        """Validate that chart builders return Plotly-like figure objects."""
        report = self._build_report()
        fig = report.build_chart_loans_by_user_type()
        self.assertTrue(hasattr(fig, "data"))


unittest.main(argv=[''], exit=False)


.....
----------------------------------------------------------------------
Ran 5 tests in 0.069s

OK
